# Building a Text-to-Text Generation System Using Transformers

## 📚 Learning Objectives

By completing this notebook, you will:
- Build text-to-text generation system
- Use transformer models
- Generate text from text
- Apply to various tasks
- Evaluate generation quality

## 🔗 Prerequisites

- ✅ Understanding of transformers
- ✅ Understanding of text generation
- ✅ Hugging Face Transformers knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 2**:
- Building a text-to-text generation system using Transformers
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 2 Practical Content

---

## Introduction

**Text-to-text generation** systems use transformer models to transform input text into output text, enabling tasks like translation, summarization, and question answering.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import numpy as np
print(f'PyTorch {torch.__version__}')
print('✅ Libraries imported!')
print('\nText-to-Text Generation with Transformers (seq2seq)')
print('=' * 60)
print('\nText-to-Text concept (T5, BART, mT5):')
print('  - Frame every NLP task as "text in → text out"')
print('  - Encoder reads input; Decoder generates output autoregressively')
print('  - This notebook builds a tiny nn.Transformer seq2seq in pure PyTorch')

# ── Toy task: reverse a sequence of digits ─────────────────────────────────
SOS, EOS, PAD = 10, 11, 12
V = 13  # vocabulary size (digits 0-9 + SOS/EOS/PAD)
torch.manual_seed(0); np.random.seed(0)

def make_pair(n=6):
    s = np.random.randint(0, 10, n).tolist()
    return s, list(reversed(s))

def collate(pairs):
    # tgt_in  = [SOS, y1, ..., yn]   length n+1
    # tgt_out = [y1,  ..., yn, EOS]  length n+1  (same length!)
    src     = torch.tensor([p[0] for p in pairs], dtype=torch.long)
    tgt_in  = torch.tensor([[SOS] + p[1] for p in pairs], dtype=torch.long)
    tgt_out = torch.tensor([p[1] + [EOS] for p in pairs], dtype=torch.long)
    return src, tgt_in, tgt_out

# ── Small Transformer seq2seq ────────────────────────────────────────────────
class Seq2SeqTransformer(nn.Module):
    def __init__(self, V, d=32, nhead=4, nlayers=2):
        super().__init__()
        self.src_emb = nn.Embedding(V, d)
        self.tgt_emb = nn.Embedding(V, d)
        self.transformer = nn.Transformer(d, nhead, nlayers, nlayers,
                                          dim_feedforward=64, batch_first=True)
        self.fc = nn.Linear(d, V)
    def forward(self, src, tgt):
        mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1))
        out = self.transformer(self.src_emb(src), self.tgt_emb(tgt),
                               tgt_mask=mask, tgt_is_causal=True)
        return self.fc(out)

model   = Seq2SeqTransformer(V)
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
pairs   = [make_pair() for _ in range(4000)]

for epoch in range(20):
    model.train(); el = 0
    for i in range(0, len(pairs), 128):
        src, tgt_in, tgt_out = collate(pairs[i:i+128])
        logits = model(src, tgt_in)          # (B, T, V)
        loss   = loss_fn(logits.reshape(-1, V), tgt_out.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step(); el += loss.item()
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1}: loss={el:.4f}')

def decode(src_seq, max_len=10):
    model.eval()
    src = torch.tensor([src_seq], dtype=torch.long)
    tgt = torch.tensor([[SOS]], dtype=torch.long)
    for _ in range(max_len):
        with torch.no_grad(): logits = model(src, tgt)
        nxt = logits[0, -1].argmax().item()
        if nxt == EOS: break
        tgt = torch.cat([tgt, torch.tensor([[nxt]])], dim=1)
    return tgt[0, 1:].tolist()

test_pairs = [make_pair() for _ in range(5)]
print('\n(seq2seq reverse task  |  input → expected → model output)')
correct = 0
for s, t in test_pairs:
    pred = decode(s); ok = pred == t; correct += ok
    print(f'  {s} → {t} | model: {pred}  {"✓" if ok else "✗"}')
print(f'Accuracy: {correct}/{len(test_pairs)}')

PyTorch 2.13.0
✅ Libraries imported!

Text-to-Text Generation with Transformers (seq2seq)

Text-to-Text concept (T5, BART, mT5):
  - Frame every NLP task as "text in → text out"
  - Encoder reads input; Decoder generates output autoregressively
  - This notebook builds a tiny nn.Transformer seq2seq in pure PyTorch


Epoch 5: loss=36.4935


Epoch 10: loss=33.2080


Epoch 15: loss=31.6847


Epoch 20: loss=30.9843

(seq2seq reverse task  |  input → expected → model output)
  [8, 5, 0, 7, 5, 2] → [2, 5, 7, 0, 5, 8] | model: [5, 7, 8, 2, 0, 5]  ✗
  [0, 2, 1, 1, 9, 2] → [2, 9, 1, 1, 2, 0] | model: [2, 1, 0, 9, 1, 2]  ✗
  [1, 5, 5, 6, 9, 2] → [2, 9, 6, 5, 5, 1] | model: [5, 2, 6, 9, 1, 5]  ✗
  [3, 8, 2, 2, 4, 5] → [5, 4, 2, 2, 8, 3] | model: [2, 2, 3, 8, 4, 5]  ✗
  [3, 5, 2, 1, 0, 3] → [3, 0, 1, 2, 5, 3] | model: [3, 3, 1, 5, 2, 0]  ✗
Accuracy: 0/5


## 🌍 Real-World Worked Example — Character-Level Text Generator

**Industry context:**
- GitHub Copilot generates code character by character using GPT-4
- ChatGPT predicts the next token based on all previous context
- Autocomplete on your phone uses a smaller version of the same idea

We build a **character-level language model** that learns to generate text token by token — the exact mechanism behind all LLMs.

In [2]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Training text ────────────────────────────────────────────────────────────
text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep"
)

chars  = sorted(set(text))
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]

SEQ_LEN = 20
X_list, y_list = [], []
for i in range(len(enc)-SEQ_LEN-1):
    X_list.append(enc[i:i+SEQ_LEN])
    y_list.append(enc[i+SEQ_LEN])
X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)

# ── LSTM Language Model ───────────────────────────────────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out,_ = self.lstm(self.embed(x))
        return self.fc(out[:,-1,:])

model   = CharLM()
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    model.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(model(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} — loss: {loss.item():.3f}")

# ── Text Generation (Greedy / Temperature Sampling) ──────────────────────
def generate(seed_str, steps=80, temperature=0.8):
    model.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = model(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = np.random.choice(len(probs), p=probs)
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

print("\n── Generated Text ──────────────────────────────────────────────")
print(generate("to be or not", steps=100))
print("\nThis is exactly how ChatGPT generates text — one token at a time.")

Epoch 0 — loss: 3.169


Epoch 50 — loss: 1.298


Epoch 100 — loss: 0.053


Epoch 150 — loss: 0.011



── Generated Text ──────────────────────────────────────────────
to be or notef a the qusearin s ake to slee ha ind the the sand tis tha cles the slings an wipher tond a natura 

This is exactly how ChatGPT generates text — one token at a time.


## 📚 References & Further Reading

**Papers:**
- Radford et al. (2019) — [GPT-2: Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

**State-of-the-Art:** GPT-4, Claude 3.5, Gemini 1.5 — all trained on trillions of tokens with transformer decoders.

## 📝 Summary

You built **autoregressive text generation** — the core mechanism behind GPT, ChatGPT, and LLaMA. The model learns to predict the next token given context. Temperature controls creativity vs coherence. Scaling this architecture to billions of parameters creates foundation models.